Cloud-only data dependency: this notebook expects access to OncDRS/cloud data paths, and either Azure AD credentials (`dfci_gpt`) or Application Default Credentials with Vertex AI permission (`vertex_ai`), depending on `PROVIDER` below.

# AVPC / NEPC Criteria Timeline Pipeline Runner

Runs the longitudinal AVPC/NEPC criteria extraction pipeline through either provider, selected with the
`PROVIDER` toggle in the parameters cell.

Pipeline steps:
1. Run `preprocessing/cli/compile_prostate_notes.py` to build `prostate_text_data.csv` (if not already built).
2. Run `preprocessing/cli/collect_nepc_notes.py` to collect triggered note snippets into an evidence TSV (provider-independent).
3. Run `tasks/longitudinal_NEPC/build_nepc_timeline.py --provider PROVIDER` to call the LLM on the evidence and write the timeline.

All run toggles default to `False`; review the printed commands and paths before enabling a step.

In [ ]:
from pathlib import Path
import os
import shlex
import subprocess
import sys


def find_repo_root(start):
    """Find the repo root (contains pyproject.toml) from the notebook directory."""
    start = Path(start).resolve()
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").is_file():
            return candidate
    raise FileNotFoundError("Could not locate the repo root (pyproject.toml)")


REPO_ROOT = find_repo_root(Path.cwd())
PYTHON = sys.executable


def shell_join(parts):
    return " ".join(shlex.quote(str(part)) for part in parts)


def run_command(parts, env=None):
    print(shell_join(parts))
    completed = subprocess.run(parts, cwd=REPO_ROOT, env=env, check=False)
    if completed.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {completed.returncode}")


def build_run_env(provider):
    """Child-process environment for the selected provider. Existing environment
    values are preserved except for the explicit notebook parameters below."""
    env = os.environ.copy()
    if provider == "vertex_ai":
        env["VERTEX_PROJECT"] = VERTEX_PROJECT
        env["VERTEX_LOCATION"] = VERTEX_LOCATION
    return env


DEFAULT_DATA_ROOT = Path(
    os.environ.get("LLM_ANNOTATIONS_DATA_PATH", "/data/gusev/USERS/jpconnor/data/LLM_annotations/")
)
DEFAULT_NOTES_CSV = DEFAULT_DATA_ROOT / "prostate_text_data.csv"

print(f"Repo root: {{REPO_ROOT}}")


In [ ]:
# Provider toggle — everything below adapts to this.
PROVIDER = "vertex_ai"          # or "dfci_gpt"

MODEL_NAME = {
    "dfci_gpt":  "gpt-4o",
    "vertex_ai": "gemini-2.5-flash-lite",
}[PROVIDER]

# Vertex AI settings (unused when PROVIDER == "dfci_gpt")
VERTEX_PROJECT = "gusevlabllm"
VERTEX_LOCATION = "us-central1"

MRNS = []
MRN_FILE = "/data/gusev/USERS/jpconnor/data/CAIA/COMPASS/mrn_lists/icd_prostate_mrn_flags.csv"
NOTES_CSV_PATH = DEFAULT_NOTES_CSV

# Repeat entries here to override the default raw OncDRS note roots.
RAW_TEXT_PATHS = []

OUTPUT_DIR = DEFAULT_DATA_ROOT / "LLM_avpc_nepc_timeline"
EVIDENCE_PATH = OUTPUT_DIR / "avpc_nepc_evidence.tsv"

# Collection settings
NOTE_TYPES = None             # e.g. ["Pathology"] — restrict scanning to these NOTE_TYPE values (e.g. Pathology Imaging)
CONTEXT_CHARS = 2000
OVERWRITE_EVIDENCE = False

# LLM extraction settings
MAX_WORKERS = 16
MAX_RETRIES = 3
LIMIT_PATIENTS = None
OVERWRITE = False

# Step toggles
RUN_COMPILE_NOTES = False
RUN_COLLECT_EVIDENCE = False  # required before extraction
RUN_EXTRACTION = False
RUN_RETRY_FAILURES = False    # rerun only patients not yet marked "ok"

RUN_ENV = build_run_env(PROVIDER)


In [ ]:
notes_csv_exists = Path(NOTES_CSV_PATH).exists()
evidence_exists = Path(EVIDENCE_PATH).exists()
timeline_path = Path(OUTPUT_DIR) / "avpc_nepc_timeline.tsv"
timeline_exists = timeline_path.exists()

print(f"Provider: {PROVIDER}")
print(f"Model: {MODEL_NAME}")
print(f"Parallel workers: {MAX_WORKERS}")
print(f"Compiled notes CSV exists: {notes_csv_exists}")
print(f"Path: {NOTES_CSV_PATH}")
print(f"Evidence TSV exists: {evidence_exists}")
print(f"Path: {EVIDENCE_PATH}")
print(f"Timeline TSV exists: {timeline_exists}")
print(f"Path: {timeline_path}")

In [ ]:
compile_notes_cmd = [
    PYTHON,
    "preprocessing/cli/compile_prostate_notes.py",
    "--output-path",
    NOTES_CSV_PATH,
]

if MRNS:
    compile_notes_cmd.extend(["--mrns", ",".join(str(mrn) for mrn in MRNS)])
if MRN_FILE is not None:
    compile_notes_cmd.extend(["--mrn-file", MRN_FILE])
if RAW_TEXT_PATHS:
    for raw_text_path in RAW_TEXT_PATHS:
        compile_notes_cmd.extend(["--raw-text-path", raw_text_path])

print(shell_join(compile_notes_cmd))

In [ ]:
if RUN_COMPILE_NOTES:
    run_command(compile_notes_cmd, env=RUN_ENV)
else:
    print("Skipping compile_prostate_notes.py")

In [ ]:
collect_evidence_cmd = [
    PYTHON,
    "preprocessing/cli/collect_nepc_notes.py",
    "--notes-csv",
    NOTES_CSV_PATH,
    "--output-dir",
    OUTPUT_DIR,
    "--context-chars",
    str(CONTEXT_CHARS),
]

if MRNS:
    collect_evidence_cmd.extend(["--mrns", ",".join(str(mrn) for mrn in MRNS)])
if MRN_FILE is not None:
    collect_evidence_cmd.extend(["--mrn-file", MRN_FILE])
if RAW_TEXT_PATHS:
    for raw_text_path in RAW_TEXT_PATHS:
        collect_evidence_cmd.extend(["--raw-text-path", raw_text_path])
if NOTE_TYPES:
    collect_evidence_cmd.extend(["--note-types", *NOTE_TYPES])

print(shell_join(collect_evidence_cmd))

In [ ]:
if RUN_COLLECT_EVIDENCE:
    run_command(collect_evidence_cmd, env=RUN_ENV)
else:
    print("Skipping collect_nepc_notes.py")

In [ ]:
run_extraction_cmd = [
    PYTHON,
    "tasks/longitudinal_NEPC/build_nepc_timeline.py",
    "--provider",
    PROVIDER,
    "--output-dir",
    OUTPUT_DIR,
    "--evidence-path",
    EVIDENCE_PATH,
    "--model",
    MODEL_NAME,
    "--max-workers",
    str(MAX_WORKERS),
    "--max-retries",
    str(MAX_RETRIES),
]

if MRNS:
    run_extraction_cmd.extend(["--mrns", ",".join(str(mrn) for mrn in MRNS)])
if MRN_FILE is not None:
    run_extraction_cmd.extend(["--mrn-file", MRN_FILE])
if LIMIT_PATIENTS is not None:
    run_extraction_cmd.extend(["--limit-patients", str(LIMIT_PATIENTS)])
if OVERWRITE:
    run_extraction_cmd.append("--overwrite")

print(shell_join(run_extraction_cmd))

In [ ]:
if RUN_EXTRACTION:
    run_command(run_extraction_cmd, env=RUN_ENV)
else:
    print("Skipping build_nepc_timeline.py")

In [ ]:
# Rerun only patients not yet marked "ok" in the processed-patients log.
# (These pipelines do not track a separate failures TSV — rerunning without
# --overwrite skips already-completed patients automatically.)
retry_cmd = [part for part in run_extraction_cmd if part != "--overwrite"]
print(shell_join(retry_cmd))

if RUN_RETRY_FAILURES:
    run_command(retry_cmd, env=RUN_ENV)
else:
    print("Skipping retry")